# CS781 Capstone — Phase 1: Mamba (selective state-space) language model

Same pipeline as the GQLA notebook — **same data sampling, same cleaning, same BPE tokenizer, same train/val split, same 512-token chunking, same test protocol (512-token prompt → 128-token continuation) and same evaluator (ROUGE-L / BLEU-4)** — with only the model swapped for Mamba (Gu & Dao, 2023, arXiv:2312.00752).

**Layout**
1. Setup + definitions (tokenizer class, cleaning functions) — always run
2. Data preparation (download → sample → clean → tokenizer → split → encode) — **skip if the cache check says CACHED** (re-uses the exact same tokens as the GQLA run, which is what you want for a fair comparison)
3. Load tokenizer + tokens, build data loaders
4. Mamba model
5. Training
6. Test-set generation + evaluation

## 1. Setup and definitions

In [ ]:
!pip install -U datasets pyarrow
!pip install -q ijson
!pip install bert-score
# !pip install causal-conv1d mamba-ssm --no-build-isolation
# NOTE: You MUST restart the runtime (Runtime -> Restart session) after this cell finishes,
# otherwise the old versions of datasets and pyarrow will still be loaded in memory.

# Optional (GPU only, needs a matching CUDA toolchain; the notebook works without it):
# !pip install mamba-ssm causal-conv1d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 29.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 663.3 kB/s eta 0:00:00


In [ ]:
import os, getpass
from datasets import load_dataset
from huggingface_hub import hf_hub_download, login

HF_DATASET_ID = "Exploration-Lab/CS-781-Capstone"

# Do NOT hard-code the token in the notebook.
# Colab: add it under the key icon (Secrets) as HF_TOKEN, enable notebook access.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token: ")


def load_legal_dataset(hf_token=None, split=None, phase=None, streaming=False):
    if hf_token is None:
        raise ValueError("Hugging Face token not found. Set HF_TOKEN or pass token explicitly.")
    return load_dataset(
        HF_DATASET_ID, phase, split=split, token=hf_token, streaming=streaming,
    )


login(token=hf_token)

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Same cache directory as the GQLA notebook, so tokenizer + tokenised data are shared.
cache_dir = '/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache'
os.makedirs(cache_dir, exist_ok=True)

import psutil
process = psutil.Process(os.getpid())

Mounted at /content/drive


In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


In [ ]:
# ============================================================
# DATA CLEANING
#   1. Unicode + whitespace normalisation
#   2. Language / script filtering (keep English, drop the rest)
#   3. De-duplication (exact + near-duplicate via MinHash-LSH)
#
# Run this AFTER the raw documents are sampled (train_val_texts)
# and BEFORE the train / val split, so that duplicates cannot
# leak from train into validation.
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text


# ------------------------------------------------------------
# 2. LANGUAGE / SCRIPT HANDLING (no external dependency)
# ------------------------------------------------------------
_LATIN = "A-Za-z\u00C0-\u024F\u1E00-\u1EFF"
_SCRIPTS = {
    "devanagari": "\u0900-\u097F", "bengali": "\u0980-\u09FF",
    "gurmukhi":   "\u0A00-\u0A7F", "gujarati": "\u0A80-\u0AFF",
    "odia":       "\u0B00-\u0B7F", "tamil":    "\u0B80-\u0BFF",
    "telugu":     "\u0C00-\u0C7F", "kannada":  "\u0C80-\u0CFF",
    "malayalam":  "\u0D00-\u0D7F",
    "arabic/urdu": "\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF",
    "cyrillic":   "\u0400-\u04FF",
    "cjk":        "\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF",
}
_SCRIPT_RES = {k: re.compile(f"[{v}]") for k, v in _SCRIPTS.items()}
_ALPHA_RE = re.compile(r"[^\W\d_]")
_LATIN_RE = re.compile(f"[{_LATIN}]")
# a run of non-Latin letters (plus the combining marks / joiners that belong to them)
_FOREIGN_RUN_RE = re.compile(
    "[" + "".join(_SCRIPTS.values()) + "]+"
    "(?:[\\s\u200c\u200d]*[" + "".join(_SCRIPTS.values()) + "]+)*"
)
_EN_WORD_RE = re.compile(r"[a-z]+")
_ENGLISH_STOP = frozenset("""
the of and to in that is was for by on with as be it at this which are from or not has have
had been an a he she they his her their its were any such under also but if all no can shall
may would
""".split())


def language_profile(text, sample_chars=3000):
    """Cheap profile from the start and middle of a document.
    Returns (latin_ratio, english_stopword_ratio, dominant_foreign_script)."""
    n = len(text)
    s = text if n <= 2 * sample_chars else (
        text[:sample_chars] + " " + text[n // 2: n // 2 + sample_chars])
    n_alpha = len(_ALPHA_RE.findall(s))
    if n_alpha == 0:
        return 0.0, 0.0, "none"
    n_latin = len(_LATIN_RE.findall(s))
    latin_ratio = n_latin / n_alpha

    words = _EN_WORD_RE.findall(s.lower())
    en_ratio = (sum(w in _ENGLISH_STOP for w in words) / len(words)) if words else 0.0

    foreign = "latin-non-english"
    if latin_ratio < 0.5:
        counts = {k: len(r.findall(s)) for k, r in _SCRIPT_RES.items()}
        best = max(counts, key=counts.get)
        foreign = best if counts[best] > 0 else "other"
    return latin_ratio, en_ratio, foreign


# ------------------------------------------------------------
# 3. NEAR-DUPLICATE DETECTION (MinHash + LSH banding)
# ------------------------------------------------------------
_WORD_RE = re.compile(r"\w+")
_MH_PRIME = np.uint64(4294967311)           # prime just above 2**32
_ROLL = np.uint64(1000003)


class NearDupIndex:
    """Streaming near-duplicate detector. The first copy seen is kept."""

    def __init__(self, num_perm=64, bands=8, threshold=0.8,
                 shingle=5, max_words=2000, seed=1):
        assert num_perm % bands == 0
        self.num_perm, self.bands = num_perm, bands
        self.rows = num_perm // bands
        self.threshold, self.shingle, self.max_words = threshold, shingle, max_words
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.b = rng.randint(0, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.sigs = []
        self.buckets = [dict() for _ in range(bands)]

    def _shingle_hashes(self, words):
        words = words[: self.max_words]
        k = self.shingle
        wh = np.fromiter((zlib.crc32(w.encode()) for w in words),
                         dtype=np.uint64, count=len(words))
        m = len(wh) - k + 1
        if m < 1:
            return wh
        sh = wh[:m].copy()
        for j in range(1, k):                 # rolling hash of k consecutive words
            sh = sh * _ROLL + wh[j: m + j]
        return (sh ^ (sh >> np.uint64(32))) & np.uint64(0xFFFFFFFF)

    def check_and_add(self, words):
        """True if a near-duplicate was already indexed; otherwise index and return False."""
        h = self._shingle_hashes(words)[None, :]
        sig = ((self.a * h + self.b) % _MH_PRIME).min(axis=1)
        r = self.rows
        keys = [hash(sig[i * r:(i + 1) * r].tobytes()) for i in range(self.bands)]
        for i, k in enumerate(keys):
            j = self.buckets[i].get(k)
            if j is not None and (self.sigs[j] == sig).mean() >= self.threshold:
                return True
        idx = len(self.sigs)
        self.sigs.append(sig)
        for i, k in enumerate(keys):
            self.buckets[i].setdefault(k, idx)
        return False


# ------------------------------------------------------------
# PIPELINE
# ------------------------------------------------------------
def clean_corpus(
    texts,
    min_words=30,                 # drop near-empty documents
    language_filter=True,         # drop non-English documents
    min_latin_ratio=0.90,         # share of letters that must be Latin script
    min_english_ratio=0.12,       # share of words that are common English words
    strip_foreign_runs=True,      # remove stray Hindi/Urdu/... fragments inside English docs
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,          # estimated Jaccard similarity above which = duplicate
    collapse_repeats=True,
    n_examples=3,
    progress_every=20_000,
):
    stats = collections.Counter()
    dropped_scripts = collections.Counter()
    examples = collections.defaultdict(list)
    seen_exact = set()
    near = NearDupIndex(threshold=near_threshold) if near_dedup else None
    kept = []

    def drop(reason, text):
        stats[reason] += 1
        if len(examples[reason]) < n_examples:
            examples[reason].append(text[:200])

    for n, raw in enumerate(texts, 1):
        stats["input_docs"] += 1
        if not isinstance(raw, str):
            drop("not_a_string", str(raw)); continue

        # 1. normalise -------------------------------------------------
        text = normalize_text(raw, collapse_repeats=collapse_repeats)
        if text != raw:
            stats["docs_changed_by_normalisation"] += 1

        # 2. cheap length pre-check (so tiny docs are reported as "too short",
        #    not mislabelled as non-English)
        if text.count(" ") + 1 < min_words:
            drop("dropped_too_short", text); continue

        # 3. language --------------------------------------------------
        if language_filter:
            latin, en, script = language_profile(text)
            if latin < min_latin_ratio:
                dropped_scripts[script] += 1
                drop("dropped_non_latin_script", text); continue
            if en < min_english_ratio:
                dropped_scripts["latin-non-english"] += 1
                drop("dropped_non_english_latin", text); continue
            if strip_foreign_runs:
                cleaned = _FOREIGN_RUN_RE.sub(" ", text)
                if cleaned != text:
                    stats["docs_with_foreign_fragments_stripped"] += 1
                    text = _WS_RE.sub(" ", cleaned).strip()

        # 4. final length check (after foreign fragments were stripped) -
        words = _WORD_RE.findall(text.lower())
        if len(words) < min_words:
            drop("dropped_too_short", text); continue

        # 5. exact duplicates ------------------------------------------
        if exact_dedup:
            key = hashlib.blake2b(" ".join(words).encode(), digest_size=16).digest()
            if key in seen_exact:
                drop("dropped_exact_duplicate", text); continue
            seen_exact.add(key)

        # 6. near duplicates -------------------------------------------
        if near is not None and near.check_and_add(words):
            drop("dropped_near_duplicate", text); continue

        kept.append(text)
        if progress_every and n % progress_every == 0:
            print(f"  processed {n:,} | kept {len(kept):,}")

    stats["kept_docs"] = len(kept)
    return kept, {"stats": stats, "dropped_scripts": dropped_scripts, "examples": examples}


def print_clean_report(report, chars_before=None, chars_after=None):
    s = report["stats"]
    n = max(s["input_docs"], 1)
    print("=" * 62)
    print(f"Input documents                : {s['input_docs']:>9,}")
    for k in ["not_a_string", "dropped_non_latin_script", "dropped_non_english_latin",
              "dropped_too_short", "dropped_exact_duplicate", "dropped_near_duplicate"]:
        if s[k]:
            print(f"  - {k:<28}: {s[k]:>9,}  ({100 * s[k] / n:5.1f}%)")
    print(f"Kept documents                 : {s['kept_docs']:>9,}  ({100 * s['kept_docs'] / n:5.1f}%)")
    print(f"Docs changed by normalisation  : {s['docs_changed_by_normalisation']:>9,}")
    print(f"Docs with foreign text stripped: {s['docs_with_foreign_fragments_stripped']:>9,}")
    if chars_before and chars_after:
        print(f"Characters                     : {chars_before:,} -> {chars_after:,}")
    if report["dropped_scripts"]:
        print("\nLanguages / scripts removed:")
        for k, v in report["dropped_scripts"].most_common():
            print(f"  {k:<20}{v:>9,}")
    for reason, ex in report["examples"].items():
        print(f"\n[{reason}] sample:")
        for e in ex:
            print("   ", repr(e[:140]))
    print("=" * 62)


In [ ]:
# Is the Phase-1 data pipeline output already on Drive (from the GQLA run)?
bpe_en_path       = os.path.join(cache_dir, "final_bpe_en.json")
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path   = os.path.join(cache_dir, "val_tokens.jsonl")

DATA_CACHED = all(os.path.exists(p) for p in [bpe_en_path, train_tokens_path, val_tokens_path])

if DATA_CACHED:
    print("CACHED: tokenizer + train/val tokens found on Drive.")
    print("-> SKIP Section 2 and go straight to Section 3.")
else:
    print("NOT CACHED: run Section 2 (download, sample, clean, tokenizer, encode).")

CACHED: tokenizer + train/val tokens found on Drive.
-> SKIP Section 2 and go straight to Section 3.


## 2. Data preparation (Phase 1) — skip if cached

Identical to the GQLA notebook: reservoir-sample each Phase-1 source → unicode/whitespace normalisation → English-only filtering → exact + MinHash-LSH near-duplicate removal → BPE (32,768 vocab) trained on the cleaned text → 95/5 split → encode to `train_tokens.jsonl` / `val_tokens.jsonl`.

In [ ]:
from huggingface_hub import hf_hub_download
import os
from huggingface_hub import login

login(token=hf_token)

repo_id = "Exploration-Lab/CS-781-Capstone"

acts_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_acts.json",
    repo_type="dataset",
)

high_courts_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_high_courts.json",
    repo_type="dataset",
)

supreme_court_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_supreme_court.json",
    repo_type="dataset",
)

tribunals_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_tribunals.json",
    repo_type="dataset",
)

print(acts_path)
print(high_courts_path)
print(supreme_court_path)
print(tribunals_path)

In [ ]:
import ijson
import random
import json
import os


def reservoir_sample_json(
    path,
    n_samples,
    seed=42,
):
    rng = random.Random(seed)

    reservoir = []
    seen = 0

    with open(path, "rb") as f:

        for record in ijson.items(f, "item"):

            if not isinstance(record, dict):
                continue

            text = record.get("text")

            if not isinstance(text, str):
                continue

            seen += 1

            if len(reservoir) < n_samples:
                reservoir.append(text)

            else:
                j = rng.randint(0, seen - 1)

                if j < n_samples:
                    reservoir[j] = text

    print(
        f"{os.path.basename(path)}: "
        f"scanned {seen:,} documents, "
        f"selected {len(reservoir):,}"
    )

    return reservoir

In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=20_000,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=350_000,
    seed=42,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=50_000,
    seed=42,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=350_000,
    seed=42,
)

In [ ]:
# rng = random.Random(42)

train_val_texts = (
    acts
    + high_courts
    + supreme_court
    + tribunals
)

# rng.shuffle(train_val_texts)

print(f"Total documents: {len(train_val_texts):,}")

In [ ]:
chars_before = sum(len(t) for t in train_val_texts)

train_val_texts, clean_report = clean_corpus(
    train_val_texts,
    min_words=30,
    language_filter=True,
    min_latin_ratio=0.90,
    min_english_ratio=0.12,
    strip_foreign_runs=True,
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,
)

chars_after = sum(len(t) for t in train_val_texts)
print_clean_report(clean_report, chars_before, chars_after)

# free the raw per-source lists (they still hold the uncleaned text in RAM)
import gc
del acts, high_courts, supreme_court, tribunals
gc.collect()

rng = random.Random(42)
# rng.shuffle(train_val_texts)
print(f"Total documents after cleaning: {len(train_val_texts):,}")

In [ ]:
vocab_size = 32768
tokenizer = BPETokenizer(vocab_size=vocab_size)

print(f"Training BPE tokenizer with vocab size {vocab_size}...")

tokenizer = BPETokenizer(
    vocab_size=vocab_size,
    reserved_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>", "<cls>"],
)
tokenizer.train(
    train_val_texts,
    save_checkpoints=[8192, 16384, 32768],
    checkpoint_dir=cache_dir
)

print("Tokenizer training finished!")

bpe_en_path = os.path.join(cache_dir, 'final_bpe_en.json')
tokenizer.save(bpe_en_path)


In [ ]:
import random

rng = random.Random(42)

indices = list(range(len(train_val_texts)))
# rng.shuffle(indices)

val_size = int(
    0.05 * len(indices)
)

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_texts = [
    train_val_texts[i]
    for i in train_indices
]

val_texts = [
    train_val_texts[i]
    for i in val_indices
]

print("Train:", len(train_texts))
print("Val:", len(val_texts))

In [ ]:
import gc
import numpy as np

def memory_safe_encode_and_save(texts, tokenizer, save_path, chunk_size=50000):
    """
    Encodes text dataset in small chunks and saves directly to disk to prevent RAM crashes.
    """
    print(f"Starting memory-safe encoding for {len(texts):,} documents...")

    # We will write the list of token IDs to a JSON-lines or partition file to keep RAM low
    with open(save_path, 'w', encoding='utf-8') as f:
        for i in range(0, len(texts), chunk_size):
            chunk = texts[i:i + chunk_size]
            print(f"  Processing chunk {i//chunk_size + 1} ({i:,} to {min(i+chunk_size, len(texts)):,})...")

            # Encode the small chunk
            encoded_chunk = tokenizer.encode_batch(
                chunk,
                add_special_tokens=True,
                num_workers=4,
                chunk_size=32
            )

            # Write to disk immediately
            for tokens in encoded_chunk:
                f.write(json.dumps(tokens) + '\n')

            # Free memory
            del chunk, encoded_chunk
            gc.collect()

    print(f"Finished encoding! Saved to {save_path}")

In [ ]:
# Define file paths for tokenized data on Google Drive
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path = os.path.join(cache_dir, "val_tokens.jsonl")

# Run memory-safe encoding
memory_safe_encode_and_save(train_texts, tokenizer, train_tokens_path)
memory_safe_encode_and_save(val_texts, tokenizer, val_tokens_path)

## 3. Load tokenizer and tokens, build data loaders

In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================

_VOCAB_SIZE = 32768
bpe_en_path = os.path.join(cache_dir, f'final_bpe_en.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

Tokenizer loaded successfully
Vocabulary size: 32,768
Number of merges: 32,504

TEXT:
The Supreme Court of India held that the right to equality is a fundamental right.

TOKEN IDS:
[2, 371, 2152, 498, 278, 961, 1189, 351, 271, 1516, 293, 9512, 306, 304, 9602, 1516, 272, 3]

TOKENS:
['<s>', 'The</w>', 'Supreme</w>', 'Court</w>', 'of</w>', 'India</w>', 'held</w>', 'that</w>', 'the</w>', 'right</w>', 'to</w>', 'equality</w>', 'is</w>', 'a</w>', 'fundamental</w>', 'right</w>', '.</w>', '</s>']

STATISTICS:
Characters : 82
Tokens     : 18
Chars/token: 4.56

TEXT:
Section 302 of the Indian Penal Code provides punishment for murder.

TOKEN IDS:
[2, 594, 4735, 278, 271, 1602, 2917, 1344, 4798, 3301, 319, 7165, 272, 3]

TOKENS:
['<s>', 'Section</w>', '302</w>', 'of</w>', 'the</w>', 'Indian</w>', 'Penal</w>', 'Code</w>', 'provides</w>', 'punishment</w>', 'for</w>', 'murder</w>', '.</w>', '</s>']

STATISTICS:
Characters : 68
Tokens     : 14
Chars/token: 4.86

TEXT:
The petitioner challenged the c

In [ ]:
# Load the tokens back from disk in a memory-light manner
def load_tokens_from_jsonl(path):
    tokens_list = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            tokens_list.append(json.loads(line))
    return tokens_list


# Define file paths for tokenized data on Google Drive
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path = os.path.join(cache_dir, "val_tokens.jsonl")

train_tokens = load_tokens_from_jsonl(train_tokens_path)
val_tokens = load_tokens_from_jsonl(val_tokens_path)
print(f"Loaded {len(train_tokens):,} train token sequences and {len(val_tokens):,} validation sequences.")

Loaded 499,996 train token sequences and 26,315 validation sequences.


In [ ]:
import torch
import os

def get_device():
    """Returns the available computing device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_checkpoint(model, optimizer, epoch, loss, filepath, best_loss=float('inf'), loss_history=None, global_step=0):
    """Saves the model and optimizer state to a checkpoint file."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history if loss_history is not None else [],
        'step': global_step, # Save global_step
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None):
    """Loads a model and optimizer from a checkpoint file."""
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, float('inf'), float('inf'), [], 0 # Added global_step as 0

    checkpoint = torch.load(filepath, map_location=get_device())
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    loss_history = checkpoint.get('loss_history', [])
    global_step = checkpoint.get('step', 0) # Load global_step

    print(f"Loaded checkpoint from '{filepath}' (Epoch {epoch}, Step {global_step})")

    return epoch, loss, best_loss, loss_history, global_step # Return global_step


In [ ]:
device = get_device()
print("Using device: ", device)

Using device:  cuda


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class LMChunkDataset(Dataset):

    def __init__(
        self,
        tokenized_documents,
        seq_len=512,
    ):
        self.documents = tokenized_documents
        self.seq_len = seq_len

        self.index = []

        for doc_idx, ids in enumerate(
            self.documents
        ):

            n = len(ids)

            if n < seq_len + 1:
                continue

            for start in range(
                0,
                n - seq_len,
                seq_len
            ):

                end = start + seq_len + 1

                if end <= n:
                    self.index.append(
                        (doc_idx, start, end)
                    )

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):

        doc_idx, start, end = self.index[idx]

        chunk = self.documents[
            doc_idx
        ][start:end]

        chunk = torch.tensor(
            chunk,
            dtype=torch.long
        )

        return {
            "input_ids": chunk[:-1],
            "labels": chunk[1:],
        }

In [ ]:
from torch.utils.data import DataLoader


seq_len = 512
batch_size = 4 # Reduced from 8

train_dataset = LMChunkDataset(
    train_tokens,
    seq_len=seq_len,
)

val_dataset = LMChunkDataset(
    val_tokens,
    seq_len=seq_len,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


del train_dataset, val_dataset

## 4. Mamba model

Implemented from the paper in plain PyTorch (no custom CUDA build needed).

| Piece | Paper | Here |
|---|---|---|
| Block | Fig. 3: gated block, `conv → SiLU → SSM`, multiplied by `SiLU(z)` branch | `MambaMixer` |
| Selection | Alg. 2 (S6): `Δ, B, C` are functions of the input | `x_proj` → `(dt, B, C)`, `dt_proj` + softplus |
| `A` init | S4D-Real, `A_n = -(n+1)` (§3.6), diagonal | `A_log` parameter, `A = -exp(A_log)` |
| `Δ` init | `softplus⁻¹(U[0.001, 0.1])` (§3.6) | bias of `dt_proj` |
| Discretisation | ZOH for `A`, simplified (Euler) for `B` — as in the reference code | `exp(ΔA)`, `Δ·B·x` |
| Scan | hardware-aware fused scan (§3.3) | chunked parallel scan in fp32 (below); optional `mamba_ssm` CUDA kernel |
| Inference | constant memory, no KV cache (§1) | `step()` carries `(conv_state, ssm_state)` |

**Scan implementation.** Within a chunk of 8 steps the recurrence `h_t = a_t·h_{t-1} + b_t` is solved with a cumulative sum of `log a_t`, then chunk states are carried across chunks with a short loop (`L/8` iterations). Per-step `log a_t` is clamped at −8 (`a ≈ 3·10⁻⁴`, effectively a full reset) so `exp(−cumsum)` cannot overflow in fp32. Gradient checkpointing per block is on by default because the scan materialises `(B, L, d_inner, d_state)` tensors.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# Optional fast CUDA kernel (pip install mamba-ssm causal-conv1d). Not required.
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    HAS_FAST_KERNEL = True
except Exception:
    selective_scan_fn = None
    HAS_FAST_KERNEL = False

USE_FAST_KERNEL = False          # set True only if mamba_ssm imported fine and you are on GPU
print("mamba_ssm CUDA kernel available:", HAS_FAST_KERNEL)

_MIN_LOG_DECAY = -8.0            # clamp on per-step log(a_t) = dt * A


# ============================================================
# Selective scan  (h_t = exp(dt_t A) * h_{t-1} + dt_t * B_t * x_t ;  y_t = C_t . h_t + D x_t)
# ============================================================
def selective_scan_ref(x, dt, A, Bm, Cm, D):
    """Sequential reference (slow; used for testing).
    x, dt: (B, L, Din)   A: (Din, N)   Bm, Cm: (B, L, N)   D: (Din,)
    returns y: (B, L, Din), h_last: (B, Din, N)"""
    Bsz, L, Din = x.shape
    N = A.shape[1]
    h = x.new_zeros(Bsz, Din, N)
    ys = []
    for t in range(L):
        dA = torch.exp((dt[:, t, :, None] * A).clamp(min=_MIN_LOG_DECAY))
        dBx = (dt[:, t] * x[:, t])[:, :, None] * Bm[:, t, None, :]
        h = dA * h + dBx
        ys.append((h * Cm[:, t, None, :]).sum(-1))
    y = torch.stack(ys, dim=1) + x * D
    return y, h


def selective_scan_chunked(x, dt, A, Bm, Cm, D, chunk=8):
    """Chunked parallel scan, same maths as selective_scan_ref (fp32)."""
    Bsz, L, Din = x.shape
    N = A.shape[1]

    pad = (-L) % chunk                       # zero padding is a no-op for the state (dt = 0 -> a = 1, b = 0)
    if pad:
        x, dt = F.pad(x, (0, 0, 0, pad)), F.pad(dt, (0, 0, 0, pad))
        Bm, Cm = F.pad(Bm, (0, 0, 0, pad)), F.pad(Cm, (0, 0, 0, pad))
    Lp = L + pad
    nc = Lp // chunk

    dA = (dt.unsqueeze(-1) * A).clamp(min=_MIN_LOG_DECAY)        # (B, Lp, Din, N), <= 0
    dBx = (dt * x).unsqueeze(-1) * Bm.unsqueeze(2)               # (B, Lp, Din, N)
    dA = dA.view(Bsz, nc, chunk, Din, N)
    dBx = dBx.view(Bsz, nc, chunk, Din, N)

    La = torch.cumsum(dA, dim=2)                                  # within-chunk cumulative log-decay, in [-8*chunk, 0]
    local = torch.exp(La) * torch.cumsum(dBx * torch.exp(-La), dim=2)   # chunk-local states (zero initial state)

    chunk_decay = torch.exp(La[:, :, -1])                         # (B, nc, Din, N)
    chunk_end = local[:, :, -1]                                   # (B, nc, Din, N)

    carry = x.new_zeros(Bsz, Din, N)
    prev = []
    for k in range(nc):                                           # only L/chunk iterations
        prev.append(carry)
        carry = chunk_decay[:, k] * carry + chunk_end[:, k]
    h_last = carry
    prev = torch.stack(prev, dim=1)                               # state entering each chunk

    h = local + torch.exp(La) * prev.unsqueeze(2)                 # (B, nc, chunk, Din, N)
    y = (h * Cm.view(Bsz, nc, chunk, 1, N)).sum(-1).reshape(Bsz, Lp, Din)[:, :L]
    y = y + x[:, :L] * D
    return y, h_last


# ============================================================
# Norm
# ============================================================
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight


# ============================================================
# Mamba mixer (the selective-SSM part of the block, Fig. 3)
# ============================================================
class MambaMixer(nn.Module):
    def __init__(
        self,
        d_model,
        d_state=16,
        d_conv=4,
        expand=2,
        dt_rank=None,
        dt_min=1e-3,
        dt_max=1e-1,
        dt_init_floor=1e-4,
        scan_chunk=8,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = expand * d_model
        self.dt_rank = math.ceil(d_model / 16) if dt_rank is None else dt_rank
        self.scan_chunk = scan_chunk

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)          # -> (x, z)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            groups=self.d_inner, padding=d_conv - 1, bias=True,
        )
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)  # -> (dt, B, C)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        # --- dt init: softplus(bias) ~ U[dt_min, dt_max] (log-uniform, as in the reference code)
        nn.init.uniform_(self.dt_proj.weight, -self.dt_rank ** -0.5, self.dt_rank ** -0.5)
        dt = torch.exp(
            torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        ).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))                               # softplus^-1
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

        # --- S4D-Real init: A_n = -(n+1)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.A_log._no_weight_decay = True
        self.D._no_weight_decay = True

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, u, return_state=False):
        """u: (B, L, d_model). If return_state, also returns (conv_state, ssm_state) for decoding."""
        Bsz, L, _ = u.shape
        x, z = self.in_proj(u).chunk(2, dim=-1)                                   # (B, L, Din) each

        conv_state = None
        if return_state:
            xt = x.transpose(1, 2)
            conv_state = (F.pad(xt, (self.d_conv - L, 0)) if L < self.d_conv
                          else xt[:, :, -self.d_conv:]).contiguous()

        x = self.conv1d(x.transpose(1, 2))[..., :L]                               # causal depthwise conv
        x = F.silu(x).transpose(1, 2)                                             # (B, L, Din)

        dt, Bm, Cm = torch.split(
            self.x_proj(x), [self.dt_rank, self.d_state, self.d_state], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt))                                         # (B, L, Din)
        A = -torch.exp(self.A_log.float())                                        # (Din, N)

        with torch.autocast(device_type=u.device.type, enabled=False):            # scan always in fp32
            xf, dtf, Bf, Cf = x.float(), dt.float(), Bm.float(), Cm.float()
            Df = self.D.float()
            if USE_FAST_KERNEL and HAS_FAST_KERNEL and u.is_cuda:
                out = selective_scan_fn(
                    xf.transpose(1, 2).contiguous(), dtf.transpose(1, 2).contiguous(), A,
                    Bf.transpose(1, 2).contiguous(), Cf.transpose(1, 2).contiguous(), Df,
                    z=None, delta_bias=None, delta_softplus=False, return_last_state=return_state,
                )
                y, h_last = (out if return_state else (out, None))
                y = y.transpose(1, 2)
            else:
                y, h_last = selective_scan_chunked(xf, dtf, A, Bf, Cf, Df, chunk=self.scan_chunk)

        y = y.to(z.dtype) * F.silu(z)                                             # gate branch
        out = self.out_proj(y)
        return (out, (conv_state, h_last)) if return_state else out

    def step(self, u, state):
        """One-token recurrent update. u: (B, d_model). O(1) time/memory per token."""
        conv_state, h = state
        x, z = self.in_proj(u).chunk(2, dim=-1)                                   # (B, Din)

        conv_state = torch.cat([conv_state[:, :, 1:], x.unsqueeze(-1)], dim=-1)   # (B, Din, d_conv)
        x = (conv_state * self.conv1d.weight.squeeze(1)).sum(-1) + self.conv1d.bias
        x = F.silu(x)

        dt, Bm, Cm = torch.split(
            self.x_proj(x), [self.dt_rank, self.d_state, self.d_state], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt)).float()
        A = -torch.exp(self.A_log.float())

        dA = torch.exp((dt.unsqueeze(-1) * A).clamp(min=_MIN_LOG_DECAY))          # (B, Din, N)
        dBx = (dt * x.float()).unsqueeze(-1) * Bm.float().unsqueeze(1)
        h = dA * h + dBx
        y = (h * Cm.float().unsqueeze(1)).sum(-1) + self.D.float() * x.float()
        y = y.to(z.dtype) * F.silu(z)
        return self.out_proj(y), (conv_state, h)


# ============================================================
# Block (pre-norm residual) and full LM
# ============================================================
class MambaBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1, **mixer_kwargs):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.mixer = MambaMixer(d_model, **mixer_kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_state=False):
        if return_state:
            y, st = self.mixer(self.norm(x), return_state=True)
            return x + self.dropout(y), st
        return x + self.dropout(self.mixer(self.norm(x)))

    def step(self, x, state):
        y, st = self.mixer.step(self.norm(x), state)
        return x + self.dropout(y), st


class MambaLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        d_state=16,
        d_conv=4,
        expand=2,
        dropout=0.1,
        pad_token_id=0,
        scan_chunk=8,
        grad_checkpointing=True,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.grad_checkpointing = grad_checkpointing

        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, dropout=dropout, d_state=d_state, d_conv=d_conv,
                       expand=expand, scan_chunk=scan_chunk)
            for _ in range(n_layers)
        ])
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        nn.init.normal_(self.token_embedding.weight, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[pad_token_id].zero_()
            for layer in self.layers:                       # residual-branch rescale (GPT-2 / reference Mamba)
                layer.mixer.out_proj.weight /= math.sqrt(n_layers)

    def forward(self, input_ids, attention_mask=None, past_key_values=None,
                use_cache=False, return_states=False):
        # attention_mask / past_key_values / use_cache are accepted only for API parity with GQLAModel.
        x = self.token_embedding(input_ids)
        states = []
        for layer in self.layers:
            if return_states:
                x, st = layer(x, return_state=True)
                states.append(st)
            elif self.grad_checkpointing and self.training and torch.is_grad_enabled():
                x = grad_checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        logits = self.lm_head(self.norm_final(x))
        return (logits, states) if return_states else logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        """Prefill the whole prompt once, then decode token-by-token with the recurrent state."""
        self.eval()
        generated_ids = input_ids.clone()
        logits, states = self.forward(input_ids, return_states=True)
        logits = logits[:, -1, :]

        for i in range(max_new_tokens):
            l = logits / temperature
            if top_k is not None:
                top_vals, top_idx = torch.topk(l, top_k)
                probs = F.softmax(top_vals, dim=-1)
                next_token_id = top_idx.gather(-1, torch.multinomial(probs, num_samples=1))
            else:
                next_token_id = torch.argmax(l, dim=-1, keepdim=True)

            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

            if eos_token_id is not None and (next_token_id == eos_token_id).all():
                break
            if i == max_new_tokens - 1:
                break

            x = self.token_embedding(next_token_id.squeeze(-1))        # (B, d_model)
            new_states = []
            for layer, st in zip(self.layers, states):
                x, st = layer.step(x, st)
                new_states.append(st)
            states = new_states
            logits = self.lm_head(self.norm_final(x))

        return generated_ids

mamba_ssm CUDA kernel available: False


In [ ]:
# ------------------------------------------------------------
# Sanity checks (run once; takes seconds). Both should print "OK".
#  1) chunked parallel scan == sequential reference scan
#  2) full-sequence forward == prefill + token-by-token recurrent decoding
# ------------------------------------------------------------
torch.manual_seed(0)
dev = device

# 1) scan
Bsz, L, Din, N = 2, 37, 24, 16                   # L deliberately not a multiple of the chunk size
x  = torch.randn(Bsz, L, Din, device=dev)
dt = torch.rand(Bsz, L, Din, device=dev) * 3.0 + 1e-3
A  = -torch.arange(1, N + 1, device=dev, dtype=torch.float32).repeat(Din, 1)
Bm = torch.randn(Bsz, L, N, device=dev)
Cm = torch.randn(Bsz, L, N, device=dev)
Dp = torch.ones(Din, device=dev)
y_ref, h_ref = selective_scan_ref(x, dt, A, Bm, Cm, Dp)
y_chk, h_chk = selective_scan_chunked(x, dt, A, Bm, Cm, Dp, chunk=8)
print("scan  max|dy| =", (y_ref - y_chk).abs().max().item(), " max|dh| =", (h_ref - h_chk).abs().max().item())
assert torch.allclose(y_ref, y_chk, atol=1e-3, rtol=1e-3) and torch.allclose(h_ref, h_chk, atol=1e-3, rtol=1e-3)
print("OK: chunked scan matches reference")

# 2) recurrent decoding
tiny = MambaLM(vocab_size=100, d_model=32, n_layers=2, dropout=0.0, pad_token_id=0).to(dev).eval()
ids = torch.randint(1, 100, (2, 37), device=dev)
with torch.no_grad():
    full = tiny(ids)                                          # (B, 37, V)
    P = 20
    _, states = tiny(ids[:, :P], return_states=True)
    worst = 0.0
    for t in range(P, ids.size(1)):
        h = tiny.token_embedding(ids[:, t])
        new_states = []
        for layer, st in zip(tiny.layers, states):
            h, st = layer.step(h, st)
            new_states.append(st)
        states = new_states
        step_logits = tiny.lm_head(tiny.norm_final(h))
        worst = max(worst, (step_logits - full[:, t]).abs().max().item())
print("decode max|dlogits| =", worst)
assert worst < 1e-3
print("OK: recurrent decoding matches full forward")
del tiny, x, dt, A, Bm, Cm

scan  max|dy| = 1.52587890625e-05  max|dh| = 9.5367431640625e-07
OK: chunked scan matches reference
decode max|dlogits| = 7.972121238708496e-07
OK: recurrent decoding matches full forward


## 5. Training

Same recipe as the GQLA notebook where it applies (batch 4 × grad-accum 16, 3 epochs, AdamW, wd 0.01, clip 1.0, checkpoints every 250 steps, resumable). Two deliberate differences:

* **Loss targets.** `GQLADataset` already returns `labels = chunk[1:]` (next-token targets), but the GQLA loss then shifts again (`logits[:, :-1]` vs `labels[:, 1:]`), so that model is trained to predict the token **two** positions ahead. Here the loss is computed directly on `(logits, labels)`.
* **Learning rate.** Mamba usually wants a higher LR than 1e-4, so the default is 3e-4 with 5 % warm-up + cosine decay to 10 %. To match the baseline recipe exactly set `learning_rate = 1e-4` and `USE_SCHEDULER = False`.

In [ ]:
import torch
import torch.nn.functional as F


def validate_lm(model, dataloader, device="cuda", use_amp=True):
    """Mean next-token cross-entropy on the validation loader.
    Dataset yields input_ids = chunk[:-1], labels = chunk[1:], so logits/labels already line up."""
    device = torch.device(device)
    model.eval()
    total_loss, num_batches = 0.0, 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=(use_amp and device.type == "cuda")):
                logits = model(input_ids)
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1)
                )

            total_loss += loss.item()
            num_batches += 1

    model.train()
    return total_loss / num_batches if num_batches > 0 else float("inf")

In [ ]:
import os
import time
import math
import torch
import torch.nn.functional as F


def train_mamba(
    model,
    dataloader,
    val_dataloader,
    optimizer,
    scheduler=None,
    device="cuda",
    epochs=1,
    max_steps=None,
    grad_accum_steps=1,
    grad_clip=1.0,
    checkpoint_dir=None,
    save_every=1000,
    log_every=100,
    tag="checkpoint",
    use_amp=True,
):
    """
    Train the Mamba LM with periodic and end-of-epoch checkpoints.

    Periodic checkpoint filenames:
        {tag}_epoch{epoch}_step{global_step}.pt

    Latest resumable checkpoint:
        {tag}.pt

    Best validation checkpoint:
        best_model.pt

    Checkpoints include:
        model, optimizer, scheduler, AMP scaler,
        epoch, batch position, global step,
        best validation loss, and loss history.
    """

    device = torch.device(device)
    model.to(device)

    if checkpoint_dir is not None:
        os.makedirs(checkpoint_dir, exist_ok=True)

    if grad_accum_steps < 1:
        raise ValueError("grad_accum_steps must be >= 1")

    if log_every < 1:
        raise ValueError("log_every must be >= 1")

    # --------------------------------------------------
    # Paths
    # --------------------------------------------------

    latest_path = (
        os.path.join(checkpoint_dir, f"{tag}.pt")
        if checkpoint_dir is not None
        else None
    )

    # --------------------------------------------------
    # Training state
    # --------------------------------------------------

    start_epoch = 0
    start_batch = 0
    global_step = 0
    best_val_loss = float("inf")
    loss_history = []

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(use_amp and device.type == "cuda"),
    )

    # --------------------------------------------------
    # Resume checkpoint
    # --------------------------------------------------

    if latest_path is not None and os.path.isfile(latest_path):
        print(f"Loading checkpoint: {latest_path}")

        ckpt = torch.load(
            latest_path,
            map_location=device,
            weights_only=False,
        )

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        if ckpt.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

        start_epoch = ckpt.get("epoch", 0)
        start_batch = ckpt.get("batch_idx", 0)
        global_step = ckpt.get("global_step", 0)
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        loss_history = ckpt.get("loss_history", [])

        # Restore RNG states when available.
        if ckpt.get("torch_rng_state") is not None:
            torch.set_rng_state(ckpt["torch_rng_state"].cpu().to(dtype=torch.uint8))

        if (
            device.type == "cuda"
            and ckpt.get("cuda_rng_state") is not None
        ):
            # Ensure each state tensor is explicitly cast to a CPU ByteTensor (uint8)
            cleaned_cuda_rng_state = [
                state.cpu().to(dtype=torch.uint8) for state in ckpt["cuda_rng_state"]
            ]
            torch.cuda.set_rng_state_all(cleaned_cuda_rng_state)

        print(
            f"Resuming from epoch={start_epoch + 1}, "
            f"batch={start_batch}, step={global_step}, "
            f"best_val_loss={best_val_loss:.4f}"
        )

    # --------------------------------------------------
    # Checkpoint helper
    # --------------------------------------------------

    def save_training_checkpoint(
        path,
        epoch_to_resume,
        batch_to_resume,
        last_loss,
    ):
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": (
                scheduler.state_dict()
                if scheduler is not None
                else None
            ),
            "scaler_state_dict": scaler.state_dict(),

            # Zero-based epoch and next batch to process.
            "epoch": epoch_to_resume,
            "batch_idx": batch_to_resume,
            "global_step": global_step,

            "loss": last_loss,
            "best_val_loss": best_val_loss,
            "loss_history": loss_history,

            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if device.type == "cuda"
                else None
            ),
        }

        # Write atomically to avoid leaving a partial checkpoint.
        temp_path = path + ".tmp"
        torch.save(checkpoint, temp_path)
        os.replace(temp_path, path)

    # --------------------------------------------------
    # Training
    # --------------------------------------------------

    model.train()

    running_loss = 0.0
    running_batches = 0
    tokens_seen = 0
    start_time = time.time()

    last_loss = float("nan")

    for epoch in range(start_epoch, epochs):

        # Only skip batches when resuming the interrupted epoch.
        batch_offset = start_batch if epoch == start_epoch else 0

        epoch_start = time.time()
        epoch_loss = 0.0
        epoch_batches = 0
        epoch_tokens = 0

        optimizer.zero_grad(set_to_none=True)

        for batch_idx, batch in enumerate(dataloader):

            # Resume at the next unprocessed batch.
            if batch_idx < batch_offset:
                continue

            input_ids = batch["input_ids"].to(
                device, non_blocking=True
            )
            labels = batch["labels"].to(
                device, non_blocking=True
            )

            _, seq_len = input_ids.shape
            batch_size = input_ids.size(0)

            # ------------------------------------------
            # Forward and loss
            # ------------------------------------------

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(use_amp and device.type == "cuda"),
            ):
                outputs = model(input_ids, use_cache=False)

                logits = (
                    outputs[0]
                    if isinstance(outputs, tuple)
                    else outputs
                )

                shift_logits = logits.contiguous()   # labels are already next-token targets
                shift_labels = labels.contiguous()

                loss = F.cross_entropy(
                    shift_logits.reshape(
                        -1, shift_logits.size(-1)
                    ),
                    shift_labels.reshape(-1),
                    ignore_index=-100,
                )

            last_loss = loss.item()

            # ------------------------------------------
            # Backward
            # ------------------------------------------

            (loss / grad_accum_steps).backward() \
                if not scaler.is_enabled() else None

            if scaler.is_enabled():
                scaler.scale(loss / grad_accum_steps).backward()

            running_loss += last_loss
            running_batches += 1
            epoch_loss += last_loss
            epoch_batches += 1

            batch_tokens = batch_size * seq_len
            tokens_seen += batch_tokens
            epoch_tokens += batch_tokens

            # ------------------------------------------
            # Optimizer step
            # ------------------------------------------

            is_last_batch = (batch_idx + 1) == len(dataloader)

            should_step = (
                (batch_idx + 1) % grad_accum_steps == 0
                or is_last_batch
            )

            if not should_step:
                continue

            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                grad_clip,
            )

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None:
                scheduler.step()

            global_step += 1

            # ------------------------------------------
            # Periodic step checkpoint
            # ------------------------------------------

            if (
                checkpoint_dir is not None
                and save_every is not None
                and save_every > 0
                and global_step % save_every == 0
            ):
                step_path = os.path.join(
                    checkpoint_dir,
                    f"{tag}_epoch{epoch + 1}_step{global_step}.pt",
                )

                # Save both the archival checkpoint and
                # the latest resumable checkpoint.

                save_training_checkpoint(
                    latest_path,
                    epoch,
                    batch_idx + 1,
                    last_loss,
                )

                print(f"Saved step checkpoint: {step_path}")

            # ------------------------------------------
            # Logging
            # ------------------------------------------

            if global_step % log_every == 0:
                avg_loss = running_loss / max(running_batches, 1)
                ppl = math.exp(min(avg_loss, 20))

                elapsed = time.time() - start_time
                tok_per_sec = tokens_seen / max(elapsed, 1e-6)

                print(
                    f"epoch={epoch + 1} | "
                    f"step={global_step:,} | "
                    f"batch={batch_idx + 1:,} | "
                    f"loss={avg_loss:.4f} | "
                    f"ppl={ppl:.2f} | "
                    f"tokens={tokens_seen / 1e6:.2f}M | "
                    f"tok/s={tok_per_sec:,.0f} | "
                    f"grad_norm={grad_norm.item():.3f}",
                    flush=True,
                )

                running_loss = 0.0
                running_batches = 0
                tokens_seen = 0
                start_time = time.time()

            # ------------------------------------------
            # Max steps
            # ------------------------------------------

            if max_steps is not None and global_step >= max_steps:

                # Persist state before returning.
                save_training_checkpoint(
                    latest_path,
                    epoch,
                    batch_idx + 1,
                    last_loss,
                ) if latest_path is not None else None

                print(f"Reached max_steps={max_steps}")
                return global_step

        # ----------------------------------------------
        # Epoch summary
        # ----------------------------------------------

        avg_train_loss = epoch_loss / max(epoch_batches, 1)

        avg_val_loss = validate_lm(
            model=model,
            dataloader=val_dataloader,
            device=device,
            use_amp=use_amp,
        )

        loss_history.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
        })

        # ----------------------------------------------
        # Best model
        # ----------------------------------------------

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

            if checkpoint_dir is not None:
                best_path = os.path.join(
                    checkpoint_dir, "best_model.pt"
                )

                save_training_checkpoint(
                    best_path,
                    epoch + 1,
                    0,
                    avg_train_loss,
                )

                print(
                    f"New best validation loss: "
                    f"{best_val_loss:.4f}"
                )

        # ----------------------------------------------
        # End-of-epoch checkpoint
        # ----------------------------------------------

        if latest_path is not None:
            save_training_checkpoint(
                latest_path,
                epoch + 1,
                0,
                avg_train_loss,
            )

            print(f"Saved epoch checkpoint: {latest_path}")

        epoch_ppl = math.exp(min(avg_train_loss, 20))

        print(
            f"\nEpoch {epoch + 1} complete\n"
            f"Train Loss      : {avg_train_loss:.4f}\n"
            f"Validation Loss : {avg_val_loss:.4f}\n"
            f"Perplexity      : {epoch_ppl:.2f}\n"
            f"Tokens          : {epoch_tokens / 1e6:.2f}M\n"
            f"Time            : "
            f"{(time.time() - epoch_start) / 60:.2f} min\n"
            f"Global Step     : {global_step:,}\n",
            flush=True,
        )

        start_batch = 0

    return global_step


In [ ]:
import gc, torch

for name in ["model", "optimizer", "scheduler", "scaler", "loss", "outputs", "logits", "batch", "input_ids", "labels"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB")

Allocated: 0.01 GB
Reserved : 0.02 GB


In [ ]:
from torch.optim import AdamW
import datetime

# ---------------- Model ----------------
vocab_size = len(tokenizer.itos)
pad_token_id = tokenizer.stoi["<pad>"]

d_model = 512
n_layers = 20           # a Mamba block ~ half a Transformer layer (paper §3.4): 20 blocks ~ 10 GQLA layers
d_state = 32             # SSM state size N (paper default)
d_conv = 16
expand = 3              # E = 2 (paper)
dropout = 0.1            # same as GQLA baseline
grad_checkpointing = True  # trade ~30 % speed for a large activation-memory saving

# ---------------- Training ----------------
epochs = 3
grad_accum_steps = 16
learning_rate = 3e-4     # set 1e-4 + USE_SCHEDULER=False to mirror the GQLA baseline exactly
weight_decay = 0.01
grad_clip = 1.0
save_every = 50
log_every = 5
max_steps=2000
USE_SCHEDULER = True
use_amp = False          # the scan is always fp32; set True for fp16 autocast on the projections

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
checkpoint_dir = os.path.join(cache_dir, "mamba_checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {checkpoint_dir}")

Checkpoints will be saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/mamba_checkpoints


In [ ]:
model = MambaLM(
    vocab_size=vocab_size,
    d_model=d_model,
    n_layers=n_layers,
    d_state=d_state,
    d_conv=d_conv,
    expand=expand,
    dropout=dropout,
    pad_token_id=pad_token_id,
    grad_checkpointing=grad_checkpointing,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized with {n_params:,} trainable parameters.")

# no weight decay on norms, biases, A_log, D
decay, no_decay = [], []
for n_, p_ in model.named_parameters():
    if not p_.requires_grad:
        continue
    (no_decay if (p_.ndim < 2 or getattr(p_, "_no_weight_decay", False)) else decay).append(p_)

optimizer = AdamW(
    [{"params": decay, "weight_decay": weight_decay},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=learning_rate, betas=(0.9, 0.95),
)

scheduler = None
if USE_SCHEDULER:
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum_steps)
    total_steps = epochs * steps_per_epoch
    warmup_steps = max(1, int(0.05 * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    print(f"Scheduler: {total_steps:,} optimizer steps, {warmup_steps:,} warm-up")

Model initialized with 86,249,984 trainable parameters.
Scheduler: 51,156 optimizer steps, 2,557 warm-up


In [ ]:
# Start (or resume) training. Re-running this cell resumes from {checkpoint_dir}/mamba_baseline.pt
print("Starting Mamba model training...")

final_step = train_mamba(
    model=model,
    dataloader=train_loader,
    val_dataloader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=epochs,
    grad_accum_steps=grad_accum_steps,
    grad_clip=grad_clip,
    checkpoint_dir=checkpoint_dir,
    save_every=save_every,
    log_every=log_every,
    max_steps=max_steps,
    use_amp=use_amp,
    tag="mamba_large",
)

print(f"Training finished at global step: {final_step:,}")

Starting Mamba model training...
Loading checkpoint: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/mamba_checkpoints/mamba_large.pt
Resuming from epoch=1, batch=15200, step=950, best_val_loss=inf
epoch=1 | step=955 | batch=15,280 | loss=4.3423 | ppl=76.88 | tokens=0.16M | tok/s=258 | grad_norm=1.436
epoch=1 | step=960 | batch=15,360 | loss=4.3579 | ppl=78.09 | tokens=0.16M | tok/s=265 | grad_norm=1.508
epoch=1 | step=965 | batch=15,440 | loss=4.3675 | ppl=78.85 | tokens=0.16M | tok/s=268 | grad_norm=1.345
epoch=1 | step=970 | batch=15,520 | loss=4.3585 | ppl=78.14 | tokens=0.16M | tok/s=265 | grad_norm=1.186
epoch=1 | step=975 | batch=15,600 | loss=4.3837 | ppl=80.13 | tokens=0.16M | tok/s=265 | grad_norm=1.196
epoch=1 | step=980 | batch=15,680 | loss=4.3298 | ppl=75.93 | tokens=0.16M | tok/s=269 | grad_norm=1.216
epoch=1 | step=985 | batch=15,760 | loss=4.3251 | ppl=75.57 | tokens=0.16M | tok/s=266 | grad_norm=1.245
epoch=1 | step=990 | batch=15,840 | loss=4.2759 |

## 6. Test-set generation and evaluation

In [ ]:
test_file_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=hf_token
)

print("Test file:", test_file_path)

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Test/test.json


In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


In [ ]:
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def run_evaluation(truth_path, prediction_path):
    truth = load_json(truth_path)
    pred = load_json(prediction_path)
    validate(truth, 'truth')
    validate(pred, 'prediction')

    if set(truth.keys()) != set(pred.keys()):
        # Find missing and extra keys for better error message
        missing_keys = set(truth.keys()) - set(pred.keys())
        extra_keys = set(pred.keys()) - set(truth.keys())
        error_msg = f"ID mismatch: "
        if missing_keys: error_msg += f"missing={len(missing_keys)} (e.g., {list(missing_keys)[:5]}), "
        if extra_keys: error_msg += f"extra={len(extra_keys)} (e.g., {list(extra_keys)[:5]}), "
        raise ValueError(error_msg.strip(', '))

    # Ensure consistent ordering for predictions and references based on truth keys
    ids_ordered = list(truth.keys())

    ps = [pred[i]['text'].split() for i in ids_ordered]
    rs = [truth[i]['text'].split() for i in ids_ordered]
    pred_texts = [pred[i]['text'].strip() for i in ids_ordered]
    ref_texts = [truth[i]['text'].strip() for i in ids_ordered]

    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError:
        device='cpu'
        print("PyTorch not found or CUDA not available, using CPU for BERTScore.")

    print("Calculating BERTScore...")
    _,_,f = bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore = float(f.mean().item())
    print(f"BERTScore: {bertscore:.4f}")

    print("Calculating ROUGE-L...")
    rouge = sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in ids_ordered)/len(ids_ordered)
    print(f"ROUGE-L: {rouge:.4f}")

    print("Calculating BLEU-4...")
    bleu = bleu4(ps, rs)
    print(f"BLEU-4: {bleu:.4f}")

    return {
        # "bertscore": bertscore,
        "rouge_l": rouge,
        "bleu_4": bleu
    }

# The main function is removed as this script will be imported and its functions called directly.


### Load best model

In [ ]:
import gc, torch, os

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

checkpoint_dir = os.path.join(cache_dir, "mamba_checkpoints")

_model_path = os.path.join(checkpoint_dir, "mamba_large.pt")

model = MambaLM(
    vocab_size=vocab_size, d_model=d_model, n_layers=n_layers,
    d_state=d_state, d_conv=d_conv, expand=expand, dropout=dropout,
    pad_token_id=pad_token_id, grad_checkpointing=False,
).to(device)

_, _, _, _, _ = load_checkpoint(_model_path, model)    # same helper as the GQLA notebook
model.eval()
print("Best model loaded successfully for prediction.")

Loaded checkpoint from '/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/mamba_checkpoints/mamba_large.pt' (Epoch 0, Step 0)
Best model loaded successfully for prediction.


### Generate predictions and ground truth

Same protocol as the GQLA notebook: 512-token prompt → 128-token greedy continuation. Unlike the GQLA `generate`, the whole prompt is fed to the model (prefill) before decoding.

In [ ]:
CONTEXT_LEN = 512
TARGET_LEN = 128
TOTAL_LEN = CONTEXT_LEN + TARGET_LEN

# The cleaning step normalises training text (quotes, dashes, whitespace). The GQLA notebook did not apply
# normalize_text() to test prompts; keep False to mirror it, True to match the training text distribution.
NORMALIZE_TEST = False

predictions_output = {}
truth_output = {}

print(f"Generating predictions for {len(test_data)} test examples...")

for example_no, (example_id, example_data) in enumerate(list(test_data.items()), 1):
    if example_no % 100 == 1 or example_no == len(test_data):
        print(f"Processing example {example_no}/{len(test_data)}: {example_id}")

    input_text = example_data['text']
    if NORMALIZE_TEST:
        input_text = normalize_text(input_text)

    token_ids = tokenizer.encode(input_text, add_special_tokens=True)

    if len(token_ids) < TOTAL_LEN:
        token_ids.extend([pad_token_id] * (TOTAL_LEN - len(token_ids)))
    else:
        token_ids = token_ids[:TOTAL_LEN]

    prompt_ids = token_ids[:CONTEXT_LEN]
    target_ids = token_ids[CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN]

    prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long).unsqueeze(0).to(device)

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=TARGET_LEN,
        temperature=1.0,
        top_k=50,                       
        eos_token_id=tokenizer.stoi['</s>'],
    )

    generated_continuation_ids = generated_ids[0, CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN].tolist()

    predictions_output[example_id] = {'text': tokenizer.decode(generated_continuation_ids)}
    truth_output[example_id] = {'text': tokenizer.decode(target_ids)}

print(f"Generated predictions for {len(predictions_output)} examples.")

Generating predictions for 1000 test examples...
Processing example 1/1000: 1
Processing example 26/1000: 26
Processing example 51/1000: 51
Processing example 76/1000: 76
Processing example 101/1000: 101
Processing example 126/1000: 126
Processing example 151/1000: 151
Processing example 176/1000: 176
Processing example 201/1000: 201
Processing example 226/1000: 226
Processing example 251/1000: 251
Processing example 276/1000: 276
Processing example 301/1000: 301
Processing example 326/1000: 326
Processing example 351/1000: 351
Processing example 376/1000: 376
Processing example 401/1000: 401
Processing example 426/1000: 426
Processing example 451/1000: 451
Processing example 476/1000: 476
Processing example 501/1000: 501
Processing example 526/1000: 526
Processing example 551/1000: 551
Processing example 576/1000: 576
Processing example 601/1000: 601
Processing example 626/1000: 626
Processing example 651/1000: 651
Processing example 676/1000: 676
Processing example 701/1000: 701
Proc

### Save predictions / truth

In [ ]:
import json

prediction_json_path = os.path.join(cache_dir, "prediction_mamba.json")
truth_json_path = os.path.join(cache_dir, "truth_mamba.json")

with open(prediction_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_output, f, ensure_ascii=False, indent=2)
print(f"Predictions saved to: {prediction_json_path}")

with open(truth_json_path, "w", encoding="utf-8") as f:
    json.dump(truth_output, f, ensure_ascii=False, indent=2)
print(f"Ground truth saved to: {truth_json_path}")

Predictions saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/prediction_mamba.json
Ground truth saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/truth_mamba.json


### Run local evaluation (ROUGE-L, BLEU-4)

In [ ]:
evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")

Calculating BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.8050
Calculating ROUGE-L...
ROUGE-L: 0.1760
Calculating BLEU-4...
BLEU-4: 0.0185

Evaluation Results:
- ROUGE-L: 0.1760
- BLEU-4: 0.0185


In [ ]:
print("prediction")
print(predictions_output['1'])

print("truth")
print(truth_output['1'])